<div dir="rtl" style="text-align: right">

‏# 00 — راه‌اندازی دوره و مجموعه‌داده

‏**زمان تخمینی:** 60–90 دقیقه
‏**پیش‌نیازها:** آشنایی پایه با Python، pandas، scikit-learn، و آمار مقدماتی.
‏**وابسته به:** هیچ‌چیز.

‏## اهداف یادگیری

‏- مسئله پیش‌بینی را در لحظه درست تصمیم‌گیری صورت‌بندی کنید.
‏- مجموعه‌داده‌های عمومی را با معیارهای فنی و آموزشی مقایسه کنید.
‏- داده UCI Bank Marketing را به‌صورت بازتولیدپذیر از فایل‌های محلی بارگذاری و cache کنید.
‏- تفاوت data catalog و data dictionary را توضیح دهید.
‏- یک catalog ساده در سطح dataset و field بسازید.

‏سؤال اصلی این دوره این است: **پیش از برقراری تماس، کدام مشتری‌ها باید برای کمپین بازاریابی سپرده مدت‌دار در اولویت قرار بگیرند؟** این یک مسئله رتبه‌بندی و انتخاب است، نه مسئله استنتاج علّی.

‏**یادداشت آموزشی:** دانشجوها معمولاً مستقیم سراغ تنظیم مدل می‌روند. اینجا عمداً آهسته‌تر پیش می‌رویم و انتخاب مجموعه‌داده، لحظه پیش‌بینی، و طراحی split را به‌عنوان گام‌های اصلی مدل‌سازی در نظر می‌گیریم.

</div>

In [ ]:
# Persian text rendering for notebook markdown and plots
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [
        "Vazirmatn",
        "Vazir",
        "IRANSans",
        "Noto Sans Arabic",
        "Noto Naskh Arabic",
        "DejaVu Sans",
    ],
    "axes.unicode_minus": False,
})

try:
    from IPython.display import HTML, display

    display(HTML('<div dir="rtl" style="text-align: right"></div>'))
except Exception:
    pass

<div dir="rtl" style="text-align: right">

‏### چرا هزینه به این شکل وزن‌دهی شده است

‏مقدار `cost` در `classification_metrics` یک ابتکار مخصوص این دوره است، نه یک فرمول آماری استاندارد. آن را این‌جا می‌گذاریم چون ارزیابی باید لحظه تصمیم‌گیری، یعنی قبل از انجام تماس، را منعکس کند.

‏منفی‌های کاذب وزن بیشتری می‌گیرند چون از دست دادن یک مشتری بالقوه به معنای از دست دادن فرصت بازاریابی کمیاب و احتمال درآمد است. مثبت‌های کاذب هم هزینه دارند، اما معمولاً فقط به معنی یک تماس تلف‌شده هستند، نه از دست رفتن یک تبدیل.

</div>

In [ ]:
import hashlib, json, os, platform, random, warnings
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
# هشدارهای سازگاری/رابط کاربری روی پیش‌بینی‌ها یا اجرای دفترچه اثر ندارند.
warnings.filterwarnings("ignore", message="X does not have valid feature names, but LGBMClassifier")
warnings.filterwarnings("ignore", message="IProgress not found.*")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (balanced_accuracy_score, brier_score_loss, confusion_matrix,
                             f1_score, log_loss, precision_score, recall_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
TARGET = "y"
LEAKAGE_COLUMNS = ["duration"]

def project_root():
    '''Return the course root when present, otherwise the notebook directory.'''
    # اگر ریشه دوره در دسترس بود همان را برگردان، وگرنه پوشه دفترچه را.
    return ROOT

def set_seed(seed=SEED):
    '''Seed Python and NumPy RNGs for reproducible notebook runs.'''
    random.seed(seed)
    np.random.seed(seed)

def fast_mode():
    '''Report whether notebooks should use the reduced fast-mode sample.'''
    # برای آزمایش‌های کامل، FAST_MODE=0 را تنظیم کنید؛ حالت لپ‌تاپ به‌صورت پیش‌فرض فعال است.
    return os.getenv("FAST_MODE", "1").lower() not in {"0", "false", "no"}

def bank_data_path():
    '''Locate the bundled Bank Marketing CSV file.'''
    # این دوره با یک مجموعه‌داده محلی همراه است؛ دفترچه‌ها هرگز به شبکه دسترسی ندارند.
    path = project_root() / "data" / "raw" / "bank-full.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"Expected the bundled Bank Marketing data at {path}. "
            "Run the notebook from the course root or place bank-full.csv there."
        )
    return path

def file_sha256(path):
    '''Compute the SHA-256 digest of a local file.'''
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def load_bank_data(include_duration=False):
    '''Load the Bank Marketing dataset and drop leakage columns by default.'''
    # داده را بارگذاری کن، y را encode کن، و مدت تماس پس از تماس را به‌صورت پیش‌فرض حذف کن.
    frame = pd.read_csv(bank_data_path(), sep=";")
    frame[TARGET] = frame[TARGET].map({"no": 0, "yes": 1}).astype("int8")
    if not include_duration:
        frame = frame.drop(columns=LEAKAGE_COLUMNS)
    return frame

def stratified_sample(frame, n, seed=SEED):
    '''Draw a label-preserving sample from a classified dataset.'''
    if n >= len(frame):
        return frame.copy()
    fractions = frame[TARGET].value_counts(normalize=True)
    counts = (fractions * n).round().astype(int)
    counts.iloc[0] += n - counts.sum()
    parts = [group.sample(n=min(counts.loc[label], len(group)),
                          random_state=seed + int(label))
             for label, group in frame.groupby(TARGET)]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

def make_splits(frame=None, reduced=None):
    '''Create deterministic stratified train, validation, and test splits.'''
    # split قطعی و stratified به نسبت 60/20/20؛ مجموعه آزمون تا دفترچه 09 قفل می‌ماند.
    from sklearn.model_selection import train_test_split
    frame = load_bank_data() if frame is None else frame
    train_val, test = train_test_split(
        frame, test_size=0.20, stratify=frame[TARGET], random_state=SEED)
    train, validation = train_test_split(
        train_val, test_size=0.25, stratify=train_val[TARGET], random_state=SEED)
    reduced = fast_mode() if reduced is None else reduced
    if reduced:
        train = stratified_sample(train, 12_000)
        validation = stratified_sample(validation, 4_000, SEED + 1)
        test = stratified_sample(test, 4_000, SEED + 2)
    return tuple(part.reset_index(drop=True) for part in (train, validation, test))

def split_xy(frame):
    '''Separate a frame into feature matrix and target vector.'''
    return frame.drop(columns=TARGET), frame[TARGET]

def feature_groups(frame):
    '''Identify numeric and categorical feature columns.'''
    features = frame.drop(columns=[TARGET], errors="ignore")
    categorical = features.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numerical = features.select_dtypes(include=np.number).columns.tolist()
    return numerical, categorical

def make_preprocessor(frame, scale_numeric=True):
    '''Build the preprocessing pipeline for numeric and categorical features.'''
    # preprocessing فقط داخل pipeline آموزش/CV مربوطه fit می‌شود.
    numerical, categorical = feature_groups(frame)
    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=10, sparse_output=True)),
    ])
    return ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numerical),
        ("categorical", categorical_pipe, categorical),
    ], sparse_threshold=0.3)

def classification_metrics(y_true, probability, threshold=0.5):
    '''Compute ranking and threshold-based classification metrics.'''
    prediction = np.asarray(probability) >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {"log_loss": log_loss(y_true, probability),
            "brier_score": brier_score_loss(y_true, probability),
            "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
            "f1": f1_score(y_true, prediction, zero_division=0),
            "precision": precision_score(y_true, prediction, zero_division=0),
            "recall": recall_score(y_true, prediction, zero_division=0),
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "cost": float(fp + 5 * fn)}

def threshold_table(y_true, probability, thresholds=None):
    '''Evaluate classification metrics across a list of decision thresholds.'''
    thresholds = np.linspace(0.05, 0.80, 76) if thresholds is None else thresholds
    return pd.DataFrame([{"threshold": float(t),
                          **classification_metrics(y_true, probability, float(t))}
                         for t in thresholds])

def add_domain_features(frame):
    '''Create domain-inspired features for the Bank Marketing dataset.'''
    result = frame.copy()
    result["was_previously_contacted"] = (result["pdays"] != -1).astype("int8")
    result["pdays_clean"] = result["pdays"].replace(-1, np.nan)
    result["contact_pressure"] = result["campaign"] / (1 + result["previous"])
    result["balance_per_age"] = result["balance"] / result["age"].clip(lower=18)
    result["age_band"] = pd.cut(result["age"], bins=[0, 29, 39, 49, 59, np.inf],
                                labels=["<30", "30s", "40s", "50s", "60+"]).astype("object")
    return result.drop(columns=["pdays"])

def environment_metadata():
    '''Collect runtime metadata for experiment tracking.'''
    import sklearn
    return {"python": platform.python_version(), "platform": platform.platform(),
            "numpy": np.__version__, "pandas": pd.__version__,
            "scikit_learn": sklearn.__version__}

def write_json(data, path):
    '''Serialize structured data to a JSON file on disk.'''
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")

set_seed(SEED)
FAST_MODE = fast_mode()
CV_FOLDS = 3 if FAST_MODE else 5
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)
print({"FAST_MODE": FAST_MODE, "CV_FOLDS": CV_FOLDS, "seed": SEED})

<div dir="rtl" style="text-align: right">

‏## کاتالوگ داده: قابل‌جست‌وجو و قابل‌حاکمیت کردن مجموعه‌داده

‏یک **کاتالوگ داده** اطلاعات سطح دارایی را نگه می‌دارد: این مجموعه‌داده چیست، چه کسی مالک آن است، از کجا آمده، آخرین بار چه زمانی به‌روزرسانی شده، و با چه سیاستی باید استفاده شود. یک **دیکشنری داده** محدودتر است و فقط فیلدهای داخل آن دارایی را توضیح می‌دهد: نام ستون، تعریف، نوع داده، دامنهٔ مقادیر، و قواعد اعتبار.

‏در این درس، تفاوت را این‌گونه خلاصه می‌کنیم:

‏1. کاتالوگ دربارهٔ «کل دارایی» است.
‏2. دیکشنری داده دربارهٔ «ستون‌ها و فیلدها» است.
‏3. کاتالوگ برای کشف، اعتماد، مالکیت، و حاکمیت استفاده می‌شود.
‏4. دیکشنری داده برای تفسیر دقیق معنا و قواعد هر فیلد استفاده می‌شود.

‏![مقایسهٔ کاتالوگ داده و دیکشنری داده: کاتالوگ در سطح دارایی عمل می‌کند و دیکشنری در سطح فیلد، تا هم حاکمیت و هم تفسیر دقیق ممکن شود.](../assets/data_catalog_learning_flow.png)

‏**یادداشت آموزشی:** کاتالوگ فقط مستندات نیست. بخشی از قرارداد مدل‌سازی است.

</div>

In [ ]:
path = bank_data_path()
raw = load_bank_data(include_duration=True)
print("cached path:", path)
print("shape:", raw.shape)
raw.head()

<div dir="rtl" style="text-align: right">

‏## کاتالوگ فیلد و دیکشنری داده

‏UCI مقدار `unknown` را یک دسته واقعی در نظر می‌گیرد؛ آن را بی‌سر‌و‌صدا به مقدار گمشده تبدیل نمی‌کنیم. `pdays == -1` یک مقدار نشانه‌ای است که یعنی مشتری قبلاً تماس نداشته است. `y=1` یعنی مشتری اشتراک را پذیرفته است. چندین فیلد به کمپین جاری مربوط‌اند؛ بنابراین باید در لحظه پیش‌بینی بررسی شوند، نه صرفاً بر اساس نام ستون‌ها.

‏کاتالوگ فیلد در ادامه، دو ویژگی حکمرانی را به خلاصهٔ معمول dtype/cardinality اضافه می‌کند:

‏- نقش هر فیلد در فرایند،
‏- و این‌که آیا فیلد در زمان پیش‌بینی در دسترس است یا نه.

‏این کار جدول را به یک feature contract تبدیل می‌کند، نه فقط یک schema dump.

</div>

In [ ]:
asset_catalog = pd.Series({
    "asset_name": "uci_bank_marketing",
    "local_path": str(path.relative_to(ROOT)),
    "source_url": "https://archive.ics.uci.edu/dataset/222/bank+marketing",
    "sha256": file_sha256(path),
    "rows": len(raw),
    "columns": raw.shape[1],
    "grain": "one row per marketing contact",
    "target": TARGET,
    "prediction_moment": "immediately before a scheduled call",
    "refresh_policy": "static course snapshot",
    "owner": "course maintainer",
    "steward": "simulated marketing-data steward",
}, name="value").to_frame()
display(asset_catalog)

<div dir="rtl" style="text-align: right">

‏## Field catalog and data dictionary

‏UCI documents `unknown` as an actual category; it is not silently converted to missing.
‏`pdays == -1` is a sentinel meaning the client was not previously contacted. `y=1`
‏means the client subscribed. Several fields describe the current campaign; availability
‏must be checked against the prediction moment rather than inferred from the column name.

‏The field catalog adds two useful governance properties:

‏- the field's role in the process,
‏- and whether the field is available at prediction time.

‏That makes the table behave like a feature contract, not just a schema dump.

</div>

In [ ]:
field_roles = {
    "age": "client attribute", "job": "client attribute", "marital": "client attribute",
    "education": "client attribute", "default": "financial attribute",
    "balance": "financial attribute", "housing": "financial attribute", "loan": "financial attribute",
    "contact": "campaign context", "day": "campaign context", "month": "campaign context",
    "duration": "post-contact measurement", "campaign": "campaign history",
    "pdays": "campaign history", "previous": "campaign history", "poutcome": "campaign history",
    TARGET: "target",
}
schema = pd.DataFrame({
    "role": [field_roles[c] for c in raw.columns],
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "unique": raw.nunique(),
    "available_at_prediction": [c != "duration" and c != TARGET for c in raw.columns],
    "example": [raw[c].iloc[0] for c in raw.columns],
})
display(schema)
print("target distribution:")
display(raw[TARGET].value_counts().rename("count").to_frame().assign(rate=lambda x: x["count"] / len(raw)))

In [ ]:
categorical = raw.select_dtypes(include="object").columns.tolist()
numeric = raw.select_dtypes(include=np.number).columns.drop(TARGET).tolist()
print("categorical:", categorical)
print("numeric:", numeric)
display(raw[categorical].nunique().sort_values().rename("cardinality").to_frame())

<div dir="rtl" style="text-align: right">

‏## ممیزی نشت: چرا `duration` باید حذف شود

‏`duration` فقط در طول یا بعد از تماس معلوم می‌شود، پس یک سیستم پیش از تماس نمی‌تواند از آن استفاده کند. این یک دام نشت کلاسیک است: ویژگی بسیار پیش‌بینی‌کننده به نظر می‌رسد، اما فقط به این دلیل که اطلاعاتِ رویدادی را که می‌خواهیم پیش‌بینی کنیم در خود دارد.

‏سلول بعدی یک pipeline ساده logistic regression را با و بدون `duration` مقایسه می‌کند. اگر امتیاز خیلی بالا برود، این یک هشدار است نه یک پیروزی استقرار.

</div>

In [ ]:
quality_checks = pd.Series({
    "target is binary and non-null": raw[TARGET].notna().all() and set(raw[TARGET].unique()) <= {0, 1},
    "age is between 18 and 100": raw["age"].between(18, 100).all(),
    "day is between 1 and 31": raw["day"].between(1, 31).all(),
    "month uses documented abbreviations": set(raw["month"]) <= set("jan feb mar apr may jun jul aug sep oct nov dec".split()),
    "pdays is -1 or non-negative": ((raw["pdays"] == -1) | (raw["pdays"] >= 0)).all(),
    "source row count is stable": len(raw) == 45_211,
}, name="passed").to_frame()
display(quality_checks)
assert quality_checks["passed"].all(), "Critical catalog quality contract failed"

<div dir="rtl" style="text-align: right">

‏**تفسیر:** `duration` معمولاً یک بهبود ظاهری چشمگیر ایجاد می‌کند چون تماس‌های موفق معمولاً طولانی‌تر هستند.

‏ریسک‌های دیگر ظریف‌ترند:

‏- `campaign` شامل تماس فعلی است،
‏- `day` و `month` تاریخ تماس فعلی را مشخص می‌کنند،
‏- و `pdays`، `previous`، `poutcome` تاریخچه تماس‌های قبلی را توصیف می‌کنند.

‏ما این فیلدها را فقط با این فرض صریح نگه می‌داریم که scoring دقیقاً پیش از یک تماس برنامه‌ریزی‌شده انجام می‌شود. لحظه تصمیم را تغییر دهید، قرارداد ویژگی هم باید تغییر کند.

</div>

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

audit = stratified_sample(raw, 12_000 if FAST_MODE else len(raw))
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

def leakage_audit(frame):
    X, y = frame.drop(columns=TARGET), frame[TARGET]
    model = Pipeline([
        ("preprocess", make_preprocessor(frame)),
        ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
    ])
    result = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=["neg_log_loss", "balanced_accuracy"],
        n_jobs=-1,
    )
    return {
        "log_loss (lower is better)": -result["test_neg_log_loss"].mean(),
        "balanced_accuracy (higher is better)": result["test_balanced_accuracy"].mean(),
    }

leakage_comparison = pd.DataFrame({
    "with_duration (invalid)": leakage_audit(audit),
    "without_duration (deployable)": leakage_audit(audit.drop(columns="duration")),
}).T
leakage_comparison

<div dir="rtl" style="text-align: right">

‏## قرارداد split

‏- **توسعه (60%)**: fit، cross-validation، انتخاب ویژگی/مدل/hyperparameter.
‏- **اعتبارسنجی (20%)**: انتخاب آستانه عملیاتی و مقایسه محدود نهایی‌ها.
‏- **آزمون (20%)**: تا دفترچه 09 قفل می‌ماند؛ فقط یک‌بار برای ارزیابی نهایی استفاده می‌شود.

‏یک split زمانی واقعی برای پیش‌بینی استقرار بهتر است، اما این فایل سال و timestamp کامل و پایدار ندارد. split گروهی برای مشتریان تکراری مفید بود، اما شناسه مشتری در داده موجود نیست.

‏**هشدار بهترین‌عملکرد:** از validation یا test برای انتخاب ویژگی‌ها، آستانه‌ها، preprocessing، یا خانواده مدل استفاده نکنید. این همان جایی است که دفترچه‌ها ناخواسته کمتر از آنچه به نظر می‌رسند قابل‌اعتماد می‌شوند.

</div>

<div dir="rtl" style="text-align: right">

‏## یک baseline عملی

‏یک دفترچه آموزشی خوب نباید در ممیزی داده متوقف شود. ما همچنین یک baseline کوچک و قابل‌استقرار می‌خواهیم که دانشجوها بتوانند آن را بفهمند، تغییر دهند، و با مدل‌های بعدی مقایسه کنند.

‏baseline زیر از این موارد استفاده می‌کند:

‏- فقط ستون‌های بدون نشت،
‏- یک pipeline استاندارد preprocessing،
‏- و logistic regression به‌عنوان یک مدل اولیه مرسوم.

‏این انتخاب عمداً ساده است. اگر یک روش پیچیده نتواند از این setup بهتر شود، باید در اضافه‌کردن پیچیدگی محتاط باشیم.

</div>

In [ ]:
clean = load_bank_data()  # duration به‌صورت پیش‌فرض حذف می‌شود
development, validation, test = make_splits(clean, reduced=FAST_MODE)
split_summary = pd.DataFrame({
    "rows": [len(development), len(validation), len(test)],
    "positive_rate": [x[TARGET].mean() for x in (development, validation, test)],
}, index=["development", "validation", "test (sealed)"])
display(split_summary)
assert "duration" not in clean.columns
assert sum(len(x) for x in make_splits(clean, reduced=False)) == len(clean)

<div dir="rtl" style="text-align: right">

‏## ارزیابی و تفسیر

‏برای مسائل نامتوازن، accuracy به‌تنهایی معمولاً گمراه‌کننده است. ما این‌ها را ترجیح می‌دهیم:

‏- log loss و Brier score برای کیفیت احتمال،
‏- precision و recall برای کاربردپذیری کلاس مثبت،
‏- balanced accuracy برای یک check ساده با توجه به توازن کلاس‌ها،
‏- و cost برای trade-off عملیاتی که واقعاً مهم است.

‏**یادداشت آموزشی:** اگر دانشجوها بپرسند چرا اینجا از test استفاده نمی‌کنیم، این نشانه خوبی است. پاسخ این است که هنوز در حال انتخاب workflow هستیم، پس روی داده توسعه می‌مانیم.

</div>

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, brier_score_loss, f1_score, log_loss
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline

def split_xy(frame):
    return frame.drop(columns=TARGET), frame[TARGET]

baseline = Pipeline([
    ("preprocess", make_preprocessor(development)),
    ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
])

X_dev, y_dev = split_xy(development)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
dev_prob = cross_val_predict(baseline, X_dev, y_dev, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

baseline_summary = pd.Series({
    "log_loss": log_loss(y_dev, dev_prob),
    "brier_score": brier_score_loss(y_dev, dev_prob),
    "balanced_accuracy@0.5": balanced_accuracy_score(y_dev, dev_prob >= 0.5),
    "f1@0.5": f1_score(y_dev, dev_prob >= 0.5, zero_division=0),
}, name="development CV")
display(baseline_summary.to_frame())

<div dir="rtl" style="text-align: right">

‏## توصیه عملی

‏از Bank Marketing به‌عنوان dataset دوره استفاده کنید، اما قرارداد پیش‌بینی را سخت‌گیرانه نگه دارید:

‏- `duration` را حذف کنید،
‏- مشخص کنید هر فیلد چه زمانی در دسترس است،
‏- از pipeline استفاده کنید تا preprocessing داخل cross-validation بماند،
‏- و test را تا notebook ارزیابی نهایی قفل نگه دارید.

‏اگر بعداً مدل را بهتر کردیم، بهبود اصلی باید از طراحی بهتر ویژگی، انتخاب آستانه، و discipline در اعتبارسنجی بیاید، نه از پنهان‌کردن نشت.

</div>

In [ ]:
dev_pred = (dev_prob >= 0.5).astype(int)
interpretation = pd.DataFrame({
    "predicted_positive_rate": [dev_pred.mean()],
    "actual_positive_rate": [y_dev.mean()],
    "precision": [precision_score(y_dev, dev_pred, zero_division=0)],
    "recall": [recall_score(y_dev, dev_pred, zero_division=0)],
}, index=["development CV"])
display(interpretation)

<div dir="rtl" style="text-align: right">

‏## توصیه‌ی عملی

‏از Bank Marketing به‌عنوان مجموعه‌داده‌ی درس استفاده کنید، اما قرارداد پیش‌بینی را سخت‌گیرانه نگه دارید:

‏- `duration` را حذف کنید،
‏- مشخص کنید هر فیلد چه زمانی در دسترس است،
‏- از pipeline استفاده کنید تا preprocessing داخل cross-validation بماند،
‏- و test set را تا دفترچه‌ی ارزیابی نهایی مهروموم نگه دارید.

‏اگر بعداً مدل را بهتر کردیم، بهبود اصلی باید از طراحی بهتر ویژگی، انتخاب آستانه و discipline در اعتبارسنجی بیاید، نه از پنهان‌کردن نشت.

</div>

<div dir="rtl" style="text-align: right">

‏## تمرین‌ها

‏1. کاتالوگ دارایی را برای یک محیط production با یک owner مشخص و freshness SLA بازنویسی کنید.
‏2. یک قانون بین‌فیلدی شامل `pdays`، `previous` و `poutcome` اضافه کنید.
‏3. لحظه‌ی پیش‌بینی را به «یک هفته قبل از کمپین» تغییر دهید و مشخص کنید کدام فیلدها دیگر در دسترس نیستند.
‏4. logistic regression را با dummy classifier جایگزین کنید و توضیح دهید چرا هنوز یک baseline مفید است.
‏5. یک معیار ارزیابی برای ذی‌نفع بازاریابی و یک معیار برای مهندس ML پیشنهاد کنید.

‏## خلاصه

‏ما یک مجموعه‌داده انتخاب کردیم، قراردادش را مستند کردیم، قواعد کیفیت پایه را بررسی کردیم، نشت را ممیزی کردیم و یک baseline کوچک و بدون نشت ساختیم. این کار بقیه‌ی درس را با زبان مشترکی برای ویژگی‌ها، splitها و ارزیابی آماده می‌کند.

‏## منابع

‏- [مجموعه‌داده UCI Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing)
‏- [مقاله‌ی مجموعه‌داده (Decision Support Systems)](https://doi.org/10.1016/j.dss.2014.03.001)
‏- [Data Catalog Vocabulary (DCAT) v3](https://www.w3.org/TR/vocab-dcat-3/)
‏- [مدیریت مدل در scikit-learn](https://scikit-learn.org/stable/model_selection.html)

</div>